<a href="https://colab.research.google.com/github/usshaa/DeepLDeploy/blob/main/Face_Recognization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

It covers dataset retrieval, transfer learning with MobileNetV3-Large, and multiple deployment optimization stages:

1. **Transfer Learning**: Backbone initialization, classification head warm-up, and fine-tuning.
2. **PyTorch Weight Pruning**: Global L1 unstructured pruning followed by fine-tuning to recover accuracy.
3. **ONNX Export & Graph Optimization**: Constant folding and dead-node removal using ONNX Runtime.
4. **Post-Training Quantization (PTQ)**: Dynamic and Static INT8 quantization using calibration data.
5. **Edge Export (TFLite INT8 / ONNX)**: Format conversion for production platforms.
6. **Empirical Benchmarking Matrix**: Comparative metrics for latency, file size, throughput, and accuracy.

---

### 1: Environment Setup & Dependencies

In [ ]:
# 1: Install required packages
!pip install -q kaggle onnx onnxruntime torchvision timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 57.2 MB/s eta 0:00:00


In [ ]:
import os
import time
import copy
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets, transforms, models
import torch.nn.utils.prune as prune

# Set deterministic seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Active Device: cuda


---

### 2: Kaggle Dataset Download & Directory Structure

In [ ]:
# 2: Kaggle Dataset Downloader & Direct Path Resolver
import json
import os

# Setup Kaggle API Token directory
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

# Option A: If you haven't uploaded kaggle.json, uncomment and add credentials:
# api_token = {"username":"YOUR_KAGGLE_USERNAME", "key":"YOUR_KAGGLE_KEY"}
# with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
#     json.dump(api_token, f)

# Secure file permissions (safely handle if file exists)
if os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    !chmod 600 ~/.kaggle/kaggle.json

# Download and unzip dataset
!kaggle datasets download -d vasukipatel/face-recognition-dataset --unzip -p ./face_dataset

Dataset URL: https://www.kaggle.com/datasets/vasukipatel/face-recognition-dataset
License(s): CC0-1.0
100% 726M/726M [00:05<00:00, 127MB/s]



In [ ]:
# 2: Point directly to Original Images where the person subfolders reside
import os

data_dir = "./face_dataset/Original Images/Original Images"

print(f"Forced Data root resolved to: {data_dir}")
print(f"Actual Person Classes: {os.listdir(data_dir)[:10]} ...")

Forced Data root resolved to: ./face_dataset/Original Images/Original Images
Actual Person Classes: ['Charlize Theron', 'Vijay Deverakonda', 'Robert Downey Jr', 'Elizabeth Olsen', 'Andy Samberg', 'Zac Efron', 'Akshay Kumar', 'Hrithik Roshan', 'Anushka Sharma', 'Priyanka Chopra'] ...


---

### 3: Data Loaders & Augmentations

In [ ]:
# 3: Data Augmentations and DataLoaders
IMG_SIZE = 224
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load full dataset via ImageFolder
full_dataset = datasets.ImageFolder(root=data_dir, transform=train_transforms)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Total Images: {len(full_dataset)} | Target Classes: {num_classes}")

# 80/20 Train/Validation Split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

# Override validation transform to remove augmentations
val_subset.dataset = copy.deepcopy(full_dataset)
val_subset.dataset.transform = val_transforms

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

Total Images: 2562 | Target Classes: 31


---

### 4: Transfer Learning (MobileNetV3-Large)

In [ ]:
# 4: Transfer Learning Architecture & Training
# MobileNetV3-Large provides high accuracy with edge-native inverted residuals & Hardswish
model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 157MB/s]


In [ ]:
# Freeze backbone initially
for param in model.features.parameters():
    param.requires_grad = False

In [ ]:
# Reconstruct classification head
in_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.Hardswish(), # Activation
    nn.Dropout(p=0.3), # 30% of our neuron inactive during training
    nn.Linear(512, num_classes) # 512 --> 31
)

In [ ]:
in_features

960

In [ ]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data)
        total += labels.size(0)
    return running_loss / total, (correct.double() / total).item()

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            total += labels.size(0)
    return (correct.double() / total).item()

In [ ]:
# Warm up classification head
print("--- Training Head ---")
for epoch in range(5):
    loss, acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}/5 | Train Acc: {acc:.4f} | Val Acc: {val_acc:.4f}")

--- Training Head ---
Epoch 1/5 | Train Acc: 0.2362 | Val Acc: 0.3138
Epoch 2/5 | Train Acc: 0.4514 | Val Acc: 0.4288
Epoch 3/5 | Train Acc: 0.5593 | Val Acc: 0.5185
Epoch 4/5 | Train Acc: 0.6179 | Val Acc: 0.5088
Epoch 5/5 | Train Acc: 0.6667 | Val Acc: 0.5906


In [ ]:
# Unfreeze top feature layers for fine-tuning
for param in model.features[-4:].parameters():
    param.requires_grad = True

In [ ]:
optimizer_ft = optim.AdamW([
    {'params': model.features[-4:].parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 5e-4}
], weight_decay=1e-4)

In [ ]:
print("--- Fine-Tuning Backbone ---")
best_acc = 0.0
for epoch in range(5):
    loss, acc = train_one_epoch(model, train_loader, optimizer_ft, criterion)
    val_acc = evaluate(model, val_loader)
    print(f"Fine-tune Epoch {epoch+1}/5 | Val Acc: {val_acc:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_fp32_model.pth")

--- Fine-Tuning Backbone ---
Fine-tune Epoch 1/5 | Val Acc: 0.6140
Fine-tune Epoch 2/5 | Val Acc: 0.6589
Fine-tune Epoch 3/5 | Val Acc: 0.6628
Fine-tune Epoch 4/5 | Val Acc: 0.6784
Fine-tune Epoch 5/5 | Val Acc: 0.6569


In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load("best_fp32_model.pth"))
fp32_acc = evaluate(model, val_loader)
print(f"Baseline FP32 Top-1 Accuracy: {fp32_acc:.4f}")

Baseline FP32 Top-1 Accuracy: 0.6784


---

### 5: Optimization Step 1 — Magnitude-Based Pruning

In [ ]:
# 5: Structured / Unstructured Magnitude Pruning
pruned_model = copy.deepcopy(model)
pruned_model.eval()

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [ ]:
# Select Conv2d and Linear weights for pruning
parameters_to_prune = []
for name, module in pruned_model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)) and "classifier.3" not in name:
        parameters_to_prune.append((module, 'weight'))

In [ ]:
# Apply 35% global L1 unstructured pruning
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.35,
)

In [ ]:
# Verify Sparsity
total_weights, zero_weights = 0, 0
for module, name in parameters_to_prune:
    total_weights += module.weight.nelement()
    zero_weights += torch.sum(module.weight == 0).item()

sparsity_pct = (zero_weights / total_weights) * 100
print(f"Calculated Global Weight Sparsity: {sparsity_pct:.2f}%")

Calculated Global Weight Sparsity: 35.00%


In [ ]:
# Quick recovery fine-tuning (1-2 epochs with small learning rate)
optimizer_prune = optim.AdamW(pruned_model.parameters(), lr=1e-5)
for epoch in range(2):
    train_one_epoch(pruned_model, train_loader, optimizer_prune, criterion)

In [ ]:
# Make pruning permanent by removing reparameterization buffers
for module, name in parameters_to_prune:
    prune.remove(module, name)

pruned_acc = evaluate(pruned_model, val_loader)
print(f"Pruned + Fine-Tuned Accuracy: {pruned_acc:.4f}")
torch.save(pruned_model.state_dict(), "pruned_model.pth")

Pruned + Fine-Tuned Accuracy: 0.6725


---

### 6: Optimization Step 2 — Export to ONNX & Graph Optimization

In [ ]:
# 6: ONNX Export, Graph Freezing, and Operator Fusion
import onnx
import onnxruntime as ort
from onnxruntime.transformers import optimizer as opt

In [ ]:
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
onnx_fp32_path = "model_pruned.onnx"

In [ ]:
# Re-export forcing dynamo=False to avoid token reordering nodes
torch.onnx.export(
    pruned_model,
    dummy_input,
    onnx_fp32_path,
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    dynamo=False  # Disables the new dynamo exporter backend causing token-reorder issues
)

/tmp/ipykernel_2512/2053395321.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [ ]:
# Validate ONNX graph integrity
onnx_model = onnx.load(onnx_fp32_path)
onnx.checker.check_model(onnx_model)
print(f"Successfully exported valid ONNX model to: {onnx_fp32_path}")

Successfully exported valid ONNX model to: model_pruned.onnx


In [ ]:
# Run ONNX Graph Optimization
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
sess_options.optimized_model_filepath = "model_optimized.onnx"
_ = ort.InferenceSession(onnx_fp32_path, sess_options, providers=['CPUExecutionProvider'])
print("Saved Graph-Fused & Frozen model to: model_optimized.onnx")

Saved Graph-Fused & Frozen model to: model_optimized.onnx


In [ ]:
# 7: Production Inference Wrapper for FP32/FP16 ONNX Runtime
inference_wrapper_code = """import json
import numpy as np
import onnxruntime as ort
from PIL import Image

class FaceRecognitionPredictor:
    def __init__(self,
                 model_path="model_optimized.onnx",
                 labels_path="labels.json"):
        self.session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.input_name = self.session.get_inputs()[0].name
        with open(labels_path, "r") as f:
            self.labels = json.load(f)

    def preprocess(self, pil_img):
        img = pil_img.convert('RGB').resize((224, 224))
        arr = np.array(img).astype(np.float32) / 255.0
        # Normalize ImageNet mean/std
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        arr = (arr - mean) / std
        arr = np.transpose(arr, (2, 0, 1)) # HWC to CHW
        return np.expand_dims(arr, axis=0) # Add batch dimension

    def predict(self, image_path):
        img = Image.open(image_path)
        tensor = self.preprocess(img)
        outputs = self.session.run(None, {self.input_name: tensor})[0]
        class_id = int(np.argmax(outputs, axis=1)[0])
        confidence = float(np.exp(outputs[0][class_id]) / np.sum(np.exp(outputs[0])))
        return {"identity": self.labels[str(class_id)], "confidence": confidence}

if __name__ == "__main__":
    predictor = FaceRecognitionPredictor()
    print("Inference service initialized and ready for production.")
"""

In [ ]:
with open("predict.py", "w") as f:
    f.write(inference_wrapper_code.strip())

In [ ]:
# Export class labels to labels.json
import json

class_dict = {str(i): name for i, name in enumerate(full_dataset.classes)}
with open("labels.json", "w") as f:
    json.dump(class_dict, f, indent=4)

print(f"Successfully exported {len(class_dict)} classes to labels.json")
print("Sample labels:", list(class_dict.items())[:5])

Successfully exported 31 classes to labels.json
Sample labels: [('0', 'Akshay Kumar'), ('1', 'Alexandra Daddario'), ('2', 'Alia Bhatt'), ('3', 'Amitabh Bachchan'), ('4', 'Andy Samberg')]


In [ ]:
# Create downloadable production deployment bundle using the accurate FP32 ONNX graph
!zip -q face_recognition_prod_bundle.zip model_optimized.onnx labels.json predict.py
print("Export complete. Download 'face_recognition_prod_bundle.zip' for production deployment.")

Export complete. Download 'face_recognition_prod_bundle.zip' for production deployment.
